# Task B · Run 2: Rebuild of the Run 1 winner

| | |
|---|---|
| **Status** | result recorded in the ledger; no log kept |
| **Outcome** | Reruns only the winning TAPT arm, with fixed seeds, in case the overnight output was lost. The ledger records 0.6013 / 0.5922 for that recipe; this copy has no separate log. |
| **Record** | [`results/experiments/task_b/02_rebuild_winner.json`](../../../results/experiments/task_b/02_rebuild_winner.json) |
| **Write-up** | [docs/experiments.md](../../../docs/experiments.md#b2) |

**Running it.** Add the organizer CSVs and `offenseval_kn.csv` to the session as a private Kaggle dataset (see `data/README.md`), switch Internet on, and use Save & Run All. The setup cell clones this repository, puts `src/` on the path and runs `python -m hastika.research.data`, which copies the files, checks their checksums and rebuilds the fixed holdouts. Research code lives under `hastika.research`. The code and run numbers match this repository; the notebook has not been re-executed since.

The cells below are the notebook as written for the run. Plans and "current best" statements in them describe that moment; the outcome above is what happened.

# HASTIKA Task B -- rebuild the winning submission

Only for the case where last night's output never published. This reruns the one arm
that won, `b_tapt` at five folds, and nothing else. About 2 hours on a T4 against the
8.5 hours the full sweep took.

Seeds and split seeds are fixed, so this reproduces the same model that scored
**0.6013** instead of rolling the dice again.

Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on. Save Version -> Save & Run All.


In [ ]:
import os, subprocess, sys, pathlib
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","main","--depth","1",
                    "https://github.com/Aaryan127/ProjectMANAS-HastikaICON2026.git", WORK], check=True)
os.chdir(WORK); os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True); print("cwd:", os.getcwd())

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
# Organizer CSVs + offenseval_kn.csv, attached as a private Kaggle dataset (see
# data/README.md): copy, verify checksums, rebuild the fixed holdouts.
subprocess.run([sys.executable, "-m", "hastika.research.data"], check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    """Streams, tees, and raises. Nothing here is allowed to fail quietly."""
    print(f"$ {' '.join(cmd)}", flush=True)
    fh = open(log,"w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Domain-adapt MuRIL (~25 min)

This is the step that was worth +2.9 points over stock MuRIL. Watch the held-out
perplexity fall across the 8 epochs.


In [ ]:
run([sys.executable,"-u","-m", "hastika.research.task_b.tapt","--out","artifacts/runs/tapt-muril"],
    log="artifacts/logs/tapt.log")


## 2. Five folds on the adapted encoder (~90 min)


In [ ]:
run([sys.executable,"-u","-m", "hastika.research.task_b.train","--tag","b_tapt_5f",
     "--model","artifacts/runs/tapt-muril","--folds","5","--seeds","42","--epochs","6"],
    log="artifacts/logs/b_tapt_5f.log")


## 3. Package

`make_submission.py` refuses to write unless the header is `id,label`, every id in
`multiclass_validation_inputs.csv` appears exactly once, and every label is one of the
six. Upload the zip it names to the Task B phase.


In [ ]:
ZIP = "/kaggle/working/b_tapt_5f.zip"
run([sys.executable,"-m", "hastika.research.common.submission","--task","b",
     "--pred","artifacts/runs/b_tapt_5f/predictions.csv","--out",ZIP])
run(["cp","artifacts/logs/b_tapt_5f.log","/kaggle/working/"])
assert os.path.exists(ZIP)
run(["unzip","-l",ZIP])

# the OOF score and the per-class report, so you can confirm it reproduced
import re
t = pathlib.Path("artifacts/logs/b_tapt_5f.log").read_text()
for line in t.splitlines():
    if "OOF macro-F1" in line or "macro avg" in line:
        print(line)
print("\nDownload b_tapt_5f.zip from the Output tab.")
